# Capa Silver : Eventos (impressions, clicks, conversions)

**Qué hace :** transforma las 3 tablas de **eventos** de bronze en tablas silver limpias dentro de `dev.silver_ads` (el schema ya lo creó `03_silver_dimensiones`). Las dimensiones se transformaron en la notebook anterior; los eventos van aparte porque comparten otra lógica: integridad de la cadena de eventos y tipos de fecha.

**Problemas medidos en bronze:**

| Problema | Medido en bronze | Solución en silver |
|---|---|---|
| Filas exactamente duplicadas (el generador repite ~0,5%) | 804 en impressions | `QUALIFY row_number()` sobre `event_key` → queda 1 |
| **Ids que se reinician en cada archivo**: un evento de noviembre y uno de octubre pueden tener el mismo `id` siendo eventos DISTINTOS | impressions tiene 150.000 ids distintos pero 160.000 eventos reales | `event_key`: huella digital (hash) de TODOS los campos del evento |
| `timestamp` guardado como TEXTO | las 3 tablas | cast a `TIMESTAMP` (y renombrado a `event_ts`, porque `timestamp` es palabra reservada) |
| Costo de impresión en $0 (~1%, valor inválido) | 1.509 en impressions | pasa a `NULL` (mismo criterio que el presupuesto de campaigns) |
| Clics huérfanos: `impression_id` que no existe | 16 clics apuntan a `9999999` | se EXCLUYEN: sin impresión no hay embudo posible |
| Conversión que apunta al id de un clic huérfano | 1 conversión (click_id = 24) | se CONSERVA: ese id también existe como clic VÁLIDO (ids repetidos entre archivos). Caso límite documentado |

**Por qué `event_key` :** el `id` NO es confiable como clave única de eventos, porque el generador lo reinicia en cada archivo ingerido. Deduplicar solo por `id` destruiría silenciosamente 10.000 eventos de noviembre. La solución: una clave sintética `event_key = xxhash64(todos los campos del evento)`. De esta forma filas idénticas producen el mismo hash (se deduplican), eventos distintos producen hashes distintos (no se pisan nunca).

**Integridad en cascada:** impressions → clicks → conversions se procesan EN ORDEN. Cada tabla excluye los registros que apuntan a un padre inexistente en la tabla silver ANTERIOR (no en bronze). Así la capa silver queda 100% navegable: todo clic tiene su impresión, toda conversión tiene su clic.

**Idempotencia:** igual que las dimensiones, todo con `MERGE` por clave (`event_key`). Re-ejecutar N veces no duplica nada.

In [0]:
%sql
-- IMPRESSIONS: el evento "se mostró un anuncio". 
-- Limpieza aplicada:
--  1) TIPO: timestamp venía como STRING -> cast a TIMESTAMP. Además lo renombramos a event_ts
--  2) NULO/INVÁLIDO: cost_usd = 0 (1.509 casos) -> NULL. Un costo de $0 no es un hecho real,es un dato faltante (mismo criterio que el presupuesto en campaigns).
--  3) CLAVE: event_key = hash de todos los campos. Es la huella digital del evento:
--     filas idénticas -> mismo hash -> se deduplican (QUALIFY row_number).
--     Eventos distintos con el mismo id -> hash distinto -> NO se pisan.
--     (NO usamos id como clave porque el generador lo reinicia en cada archivo.)

CREATE TABLE IF NOT EXISTS dev.silver_ads.impressions (
  event_key BIGINT  NOT NULL COMMENT 'Hash de todos los campos: clave única del evento',
  id        BIGINT  NOT NULL COMMENT 'Id original del archivo (NO es único entre archivos)',
  ad_id     BIGINT          COMMENT 'FK a silver.ads',
  event_ts  TIMESTAMP      COMMENT 'Momento de la impresión (antes: string)',
  country   STRING         COMMENT 'País donde se mostró',
  cost_usd  DOUBLE         COMMENT 'Costo de la impresión (NULL = dato faltante, era $0)'
) COMMENT 'impressions limpio: sin duplicados, fechas reales, clave sintética event_key';

MERGE INTO dev.silver_ads.impressions AS t
USING (
  SELECT
    xxhash64(concat_ws('|',
      cast(id   as string),
      cast(ad_id as string),
      timestamp,
      coalesce(trim(country), ''),
      coalesce(cast(cost_usd as string), '')
    )) AS event_key,
    id,
    ad_id,
    cast(timestamp AS TIMESTAMP) AS event_ts,
    trim(country) AS country,
    CASE WHEN cost_usd = 0 THEN NULL ELSE cost_usd END AS cost_usd
  FROM dev.bronze_ads.bronze_impressions
  QUALIFY row_number() OVER (PARTITION BY event_key ORDER BY event_key) = 1  -- 1 fila por evento real
) AS s
ON t.event_key = s.event_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
160000,160000,0,0


In [0]:
%sql
-- CLICKS: el evento "se hizo clic sobre una impresión".
-- Limpieza aplicada:
--  1) HUÉRFANOS: 16 clics apuntan a impression_id = 9999999, una impresión que NO existe.
--     Sin impresión no hay embudo: se EXCLUYEN (no se marcan, se sacan; están en bronze si
--     algún día se quiere auditar). La regla NO es "excluir 9999999" hardcodeado: se chequea
--     contra las impresiones VÁLIDAS de silver
--  2) TIPO: timestamp string -> event_ts TIMESTAMP.
--  3) CLAVE: mismo criterio de event_key (hash de todos los campos).

CREATE TABLE IF NOT EXISTS dev.silver_ads.clicks (
  event_key     BIGINT  NOT NULL COMMENT 'Hash de todos los campos: clave única del evento',
  id            BIGINT  NOT NULL COMMENT 'Id original del archivo (NO es único entre archivos)',
  impression_id BIGINT          COMMENT 'FK a silver.impressions (garantizado existente)',
  event_ts      TIMESTAMP      COMMENT 'Momento del clic (antes: string)',
  cost_usd      DOUBLE         COMMENT 'Costo del clic'
) COMMENT 'clicks limpio: sin huérfanos, fechas reales, clave event_key';

MERGE INTO dev.silver_ads.clicks AS t
USING (
  SELECT
    xxhash64(concat_ws('|',
      cast(id as string),
      cast(impression_id as string),
      timestamp,
      coalesce(cast(cost_usd as string), '')
    )) AS event_key,
    id,
    impression_id,
    cast(timestamp AS TIMESTAMP) AS event_ts,
    cost_usd
  FROM dev.bronze_ads.bronze_clicks k
  WHERE k.impression_id IN (SELECT id FROM dev.silver_ads.impressions)  -- solo clics con impresión válida
  QUALIFY row_number() OVER (PARTITION BY event_key ORDER BY event_key) = 1
) AS s
ON t.event_key = s.event_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3184,3184,0,0


In [0]:
%sql
-- CONVERSIONS: el evento que genera dinero. Es la punta del embudo.
-- Limpieza aplicada:
--  1) INTEGRIDAD EN CASCADA: la regla es "toda conversión debe tener su clic VÁLIDO en silver.clicks".
--     Caso límite medido: 1 conversión apunta al click_id 24, que en bronze es a la vez un clic
--     huérfano Y dos clics válidos (el id se repite entre archivos). Como existe un clic válido con
--     ese id, la conversión se CONSERVA. Es la consecuencia de ids no únicos: la referencia es
--     ambigua, pero el camino válido existe, así que no se castiga al dato.
--  2) TIPO: timestamp string -> event_ts TIMESTAMP.
--  3) VALOR: value_usd se revisó -> no hay nulos ni valores inválidos (0 casos), se conserva tal cual.
--  4) CLAVE: event_key con el mismo criterio de hash.

CREATE TABLE IF NOT EXISTS dev.silver_ads.conversions (
  event_key BIGINT  NOT NULL COMMENT 'Hash de todos los campos: clave única del evento',
  id        BIGINT  NOT NULL COMMENT 'Id original del archivo (NO es único entre archivos)',
  click_id  BIGINT          COMMENT 'FK a silver.clicks (garantizado existente)',
  event_ts  TIMESTAMP      COMMENT 'Momento de la conversión (antes: string)',
  value_usd DOUBLE         COMMENT 'Ingresos generados'
) COMMENT 'conversions limpio: cadena íntegra hasta la impresión, fechas reales';

MERGE INTO dev.silver_ads.conversions AS t
USING (
  SELECT
    xxhash64(concat_ws('|',
      cast(id as string),
      cast(click_id as string),
      timestamp,
      coalesce(cast(value_usd as string), '')
    )) AS event_key,
    id,
    click_id,
    cast(timestamp AS TIMESTAMP) AS event_ts,
    value_usd
  FROM dev.bronze_ads.bronze_conversions c
  WHERE c.click_id IN (SELECT id FROM dev.silver_ads.clicks)  -- solo conversiones con clic válido
  QUALIFY row_number() OVER (PARTITION BY event_key ORDER BY event_key) = 1
) AS s
ON t.event_key = s.event_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
256,256,0,0


In [0]:
%sql
-- VERIFICACIÓN 1 → conteos y dedup.
-- Valores esperados (medidos en bronze antes de limpiar):
--   impressions: 160.000 eventos reales (bronze tenía 160.804 filas = 160.000 + 804 duplicadas exactas)
--   clicks:      3.184  (3.200 - 16 huérfanos)
--   conversions:   256  (todas tienen un clic válido; ver el caso límite del id 24 en su celda)
SELECT 'impressions' AS tabla, COUNT(*) AS filas, COUNT(DISTINCT event_key) AS eventos_unicos
FROM dev.silver_ads.impressions
UNION ALL
SELECT 'clicks', COUNT(*), COUNT(DISTINCT event_key) FROM dev.silver_ads.clicks
UNION ALL
SELECT 'conversions', COUNT(*), COUNT(DISTINCT event_key) FROM dev.silver_ads.conversions;

tabla,filas,eventos_unicos
impressions,160000,160000
clicks,3184,3184
conversions,256,256


In [0]:
%sql
-- VERIFICACIÓN 2 → reglas de limpieza e integridad de la cadena:
--   cost_null_impresiones = 1.502  (bronze tenía 1.509 filas en $0, pero 7 eran duplicadas exactas
--                                   que el dedup ya eliminó; los NULL no desaparecieron: quedan 1.502)
--   clics_sin_impresion   = 0      (la regla anti-huérfanos funcionó)
--   conv_sin_clic         = 0      (la integridad en cascada funcionó)
--   fechas_no_timestamp   = 0      (todas las fechas ya son TIMESTAMP, no texto)
SELECT
  (SELECT COUNT(*) FROM dev.silver_ads.impressions WHERE cost_usd IS NULL) AS cost_null_impresiones,
  (SELECT COUNT(*) FROM dev.silver_ads.clicks c
     WHERE NOT EXISTS (SELECT 1 FROM dev.silver_ads.impressions i WHERE i.id = c.impression_id)) AS clics_sin_impresion,
  (SELECT COUNT(*) FROM dev.silver_ads.conversions v
     WHERE NOT EXISTS (SELECT 1 FROM dev.silver_ads.clicks k WHERE k.id = v.click_id)) AS conv_sin_clic,
  (SELECT COUNT(*) FROM dev.silver_ads.impressions WHERE event_ts IS NULL) AS fechas_null;

cost_null_impresiones,clics_sin_impresion,conv_sin_clic,fechas_null
1502,0,0,0


In [0]:
%sql
-- VERIFICACIÓN 3 → por qué event_key fue necesario (la prueba del hallazgo):
--   ids_originales_distintos = 150.000  PERO  filas = 160.000.
--   Si hubiéramos deduplicado por id, habríamos perdido 10.000 eventos de noviembre
--   (que comparten id con eventos de octubre porque el generador reinicia el contador por archivo).
SELECT
  COUNT(DISTINCT id) AS ids_originales_distintos,
  COUNT(*)          AS filas_reales,
  COUNT(*) - COUNT(DISTINCT id) AS eventos_que_se_hubieran_perdido
FROM dev.silver_ads.impressions;

ids_originales_distintos,filas_reales,eventos_que_se_hubieran_perdido
150000,160000,10000
